# Dual-SR830 frequency and current–voltage sweep browser

Read-only analysis for standalone frequency and excitation sweep JSON/JSONL files. Each scan renders six separate twin-axis figures: Vxx and Vxy for harmonic orders h1, h2, and h3. The left axis is SR830 R (voltage magnitude) and the right axis is measured phase. Missing harmonic data is labeled explicitly; no values are inferred.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from attodry_control.commissioning_analysis import (
    ExcitationPathResistance,
    browse_and_load_commissioning_file,
    discover_commissioning_records,
    export_commissioning_csv,
    load_sweep_sample_files,
    plot_six_role_harmonic_sweeps,
    summarize_commissioning_file,
)

working_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    working_directory.parent
    if working_directory.name.lower() == 'notebooks'
    else working_directory
)
DATA_DIRECTORY = PROJECT_ROOT / 'run_data' / 'commissioning'
DATA_DIRECTORY

## Start here: filters, Browse, and current calibration

Use the **Browse…** button to open the native Windows file dialog. The selected file updates the data directory so its paired frequency/excitation record can be found automatically. The `completed` checkbox is on by default; deselect it only for an explicit audit. Change the three resistance values in this cell when the physical excitation path changes: the current is always `SINE OUT RMS voltage / (external series + SR830 output + approximate device resistance)`.

In [ ]:
completed_only_widget = widgets.Checkbox(
    value=True,
    description='Only completed records',
)
sample_status_widget = widgets.SelectMultiple(
    options=('clean', 'problem', 'unlocked', 'overload', 'instrument_error'),
    value=('clean',),
    description='Formal samples',
)
include_rejected_widget = widgets.Checkbox(
    value=False,
    description='Allow rejected audit records',
)
browse_button = widgets.Button(
    description='Browse…',
    icon='folder-open',
    button_style='info',
)
selected_path_widget = widgets.Text(
    value='',
    description='Selected file',
    layout=widgets.Layout(width='85%'),
)
browse_message = widgets.HTML('Click Browse… to select a sweep JSON file.')

def _browse_file(_):
    global DATA_DIRECTORY
    try:
        browsed = browse_and_load_commissioning_file(DATA_DIRECTORY)
    except Exception as exc:
        browse_message.value = f'<b>Browse failed:</b> {exc}'
        return
    if browsed is None:
        browse_message.value = 'Browse cancelled.'
        return
    selected_path, _ = browsed
    selected_path_widget.value = str(selected_path)
    DATA_DIRECTORY = selected_path.parent
    summary = summarize_commissioning_file(selected_path)
    browse_message.value = (
        f'<b>Selected:</b> {selected_path.name} '        f'({summary.scan_type}, {summary.record_status}). '        'Run the catalog and plot cells below to refresh the figures.'
    )

browse_button.on_click(_browse_file)
display(widgets.VBox([
    widgets.HBox([browse_button, completed_only_widget, include_rejected_widget]),
    sample_status_widget,
    selected_path_widget,
    browse_message,
]))

RECORD_STATUSES = {'completed'} if completed_only_widget.value else None
SAMPLE_STATUSES = set(sample_status_widget.value)
INCLUDE_REJECTED = include_rejected_widget.value

# Change these three values to match the complete physical excitation path.
EXTERNAL_SERIES_RESISTANCE_OHM = 100_000.0
SR830_OUTPUT_RESISTANCE_OHM = 50.0
APPROXIMATE_DEVICE_RESISTANCE_OHM = 500.0
EXCITATION_PATH = ExcitationPathResistance(
    external_series_resistance_ohm=EXTERNAL_SERIES_RESISTANCE_OHM,
    sr830_output_resistance_ohm=SR830_OUTPUT_RESISTANCE_OHM,
    approximate_device_resistance_ohm=APPROXIMATE_DEVICE_RESISTANCE_OHM,
)

# Leave empty to use the newest completed record of each type. Add h2/h3 paths here
# when those sweeps are available; do not combine unrelated scan conditions.
FREQUENCY_PATHS = ()
EXCITATION_PATHS = ()

{
    'total_path_resistance_ohm': EXCITATION_PATH.total_resistance_ohm,
    'filters': {
        'record_statuses': RECORD_STATUSES,
        'sample_statuses': SAMPLE_STATUSES,
        'include_rejected': INCLUDE_REJECTED,
    },
}

## Filtered catalog

The catalog is newest-first. Toggle the completed checkbox, choose formal-sample statuses, then rerun this cell to refresh the catalog. A Browse selection changes the data directory and replaces the matching default scan; manually supplied path lists can hold separate h1/h2/h3 records from the same scan condition.

In [ ]:
RECORD_STATUSES = {'completed'} if completed_only_widget.value else None
SAMPLE_STATUSES = set(sample_status_widget.value)
INCLUDE_REJECTED = include_rejected_widget.value

catalog = discover_commissioning_records(
    DATA_DIRECTORY,
    record_statuses=RECORD_STATUSES,
    scan_types={'frequency', 'excitation'},
)
[
    {
        'file': item.path.name,
        'scan': item.scan_type,
        'status': item.record_status,
        'samples': item.sample_count,
        'problem_samples': item.problem_count,
        'error': item.error,
    }
    for item in catalog
]

## Load selected formal samples

The loader excludes transition and cleanup payloads. It refuses rejected records unless `INCLUDE_REJECTED=True`. Frequency and excitation records stay separate.

In [ ]:
completed_catalog = discover_commissioning_records(
    DATA_DIRECTORY,
    record_statuses={'completed'},
    scan_types={'frequency', 'excitation'},
)
frequency_record = next(
    (item for item in completed_catalog if item.scan_type == 'frequency'), None
)
excitation_record = next(
    (item for item in completed_catalog if item.scan_type == 'excitation'), None
)
if not FREQUENCY_PATHS and frequency_record is None:
    raise FileNotFoundError('A completed frequency record is missing.')
if not EXCITATION_PATHS and excitation_record is None:
    raise FileNotFoundError('A completed excitation record is missing.')

frequency_paths = tuple(FREQUENCY_PATHS) or (frequency_record.path,)
excitation_paths = tuple(EXCITATION_PATHS) or (excitation_record.path,)
selected_path = (
    Path(selected_path_widget.value) if selected_path_widget.value else None
)
if selected_path is not None:
    selected_summary = summarize_commissioning_file(selected_path)
    if selected_summary.scan_type == 'frequency':
        frequency_paths = (selected_path,)
    elif selected_summary.scan_type == 'excitation':
        excitation_paths = (selected_path,)
    else:
        raise ValueError('Browse selection must be a frequency or excitation sweep.')

frequency_rows = load_sweep_sample_files(
    frequency_paths,
    include_rejected=INCLUDE_REJECTED,
    sample_statuses=SAMPLE_STATUSES,
)
excitation_rows = load_sweep_sample_files(
    excitation_paths,
    include_rejected=INCLUDE_REJECTED,
    sample_statuses=SAMPLE_STATUSES,
)
{
    'frequency_files': frequency_paths,
    'frequency_samples': len(frequency_rows),
    'excitation_files': excitation_paths,
    'excitation_samples': len(excitation_rows),
}

## Six frequency figures and six current–voltage figures

Frequency figures use a logarithmic frequency axis and state the SINE OUT-derived RMS current in their titles. Current–voltage figures use the same calculated current on the x axis. Each figure keeps voltage magnitude and phase on separate y axes.

In [ ]:
frequency_figures = plot_six_role_harmonic_sweeps(
    frequency_rows,
    excitation_path=EXCITATION_PATH,
)
current_voltage_figures = plot_six_role_harmonic_sweeps(
    excitation_rows,
    excitation_path=EXCITATION_PATH,
)

for scan_name, figures in (
    ('frequency', frequency_figures),
    ('current_voltage', current_voltage_figures),
):
    print(f'{scan_name}: {len(figures)} figures')
    for (role, harmonic), figure in figures.items():
        display(figure)
        plt.close(figure)

## Optional export

No files are written unless `SAVE_OUTPUTS=True`. Exports are placed under the ignored analysis-output directory.

In [ ]:
SAVE_OUTPUTS = False
OUTPUT_DIRECTORY = PROJECT_ROOT / 'analysis_output' / 'sr830_commissioning'
if SAVE_OUTPUTS:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    export_commissioning_csv(
        frequency_rows, OUTPUT_DIRECTORY / 'frequency_samples.csv'
    )
    export_commissioning_csv(
        excitation_rows, OUTPUT_DIRECTORY / 'excitation_samples.csv'
    )
    for scan_name, figures in (
        ('frequency', frequency_figures),
        ('current_voltage', current_voltage_figures),
    ):
        for (role, harmonic), figure in figures.items():
            stem = f'{scan_name}_{role}_h{harmonic}'
            figure.savefig(OUTPUT_DIRECTORY / f'{stem}.png', dpi=200)
            figure.savefig(OUTPUT_DIRECTORY / f'{stem}.pdf')
    display(OUTPUT_DIRECTORY)